# 01 Signal Propagation

This notebook builds the first WiFiGhost simulation layer before CSI enters the picture.

The model is intentionally small:

```text
received signal = direct path + person-reflected path + noise
```

It simulates transmitter position, receiver position, person position, a direct path, a reflected path, an RSSI-like signal, and a baseline-relative motion score.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

C = 299_792_458
FREQ_HZ = 2.4e9
WAVELENGTH_M = C / FREQ_HZ

WAVELENGTH_M

## Room and Device Geometry

Use a small 2D room first. Height, antenna pattern, polarization, and wall reflections can come later.

In [ ]:
room = {"width_m": 5.0, "depth_m": 4.0}

tx = np.array([0.6, 2.0])
rx = np.array([4.4, 2.0])

# Person walks across the central sensing region, then exits it.
n_samples = 500
time_s = np.linspace(0, 20, n_samples)
person_x = np.linspace(1.0, 4.0, n_samples)
person_y = 1.15 + 0.7 * np.sin(2 * np.pi * time_s / time_s[-1])
person = np.column_stack([person_x, person_y])

tx, rx, person[:3]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, room["width_m"], room["width_m"], 0, 0], [0, 0, room["depth_m"], room["depth_m"], 0], color="0.2")
ax.scatter(*tx, s=120, marker="^", label="TX")
ax.scatter(*rx, s=120, marker="s", label="RX")
ax.plot(person[:, 0], person[:, 1], color="tab:orange", label="person path")
ax.scatter(*person[0], s=70, color="tab:green", label="person start")
ax.scatter(*person[-1], s=70, color="tab:red", label="person end")
ax.plot([tx[0], rx[0]], [tx[1], rx[1]], "--", color="tab:blue", alpha=0.7, label="direct path")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.2, room["width_m"] + 0.2)
ax.set_ylim(-0.2, room["depth_m"] + 0.2)
ax.set_xlabel("x position (m)")
ax.set_ylabel("y position (m)")
ax.set_title("WiFiGhost first propagation scene")
ax.legend(loc="upper right")
plt.show()

## Direct and Reflected Paths

The direct path is fixed. The reflected path changes as the person moves, using the person as a simple moving scatterer.

In [ ]:
def distance(a, b):
    return np.linalg.norm(a - b, axis=-1)


def free_space_amplitude(distance_m, reference_distance_m=1.0):
    safe_distance = np.maximum(distance_m, reference_distance_m)
    return reference_distance_m / safe_distance


def complex_path(distance_m, attenuation, wavelength_m=WAVELENGTH_M):
    phase_rad = -2 * np.pi * distance_m / wavelength_m
    return attenuation * np.exp(1j * phase_rad)


direct_distance_m = distance(tx, rx)
reflected_distance_m = distance(tx, person) + distance(person, rx)

direct_attenuation = free_space_amplitude(direct_distance_m)
reflected_attenuation = 0.35 * free_space_amplitude(reflected_distance_m)

direct_signal = complex_path(direct_distance_m, direct_attenuation)
reflected_signal = complex_path(reflected_distance_m, reflected_attenuation)

direct_distance_m, reflected_distance_m.min(), reflected_distance_m.max()

In [ ]:
noise_power = 0.012
noise = noise_power * (rng.normal(size=n_samples) + 1j * rng.normal(size=n_samples))

received = direct_signal + reflected_signal + noise
received_power = np.abs(received) ** 2
rssi_like_db = 10 * np.log10(received_power + 1e-12)

rssi_like_db[:5]

## Motion Score

Use the first two seconds as a baseline. The score is the absolute RSSI-like deviation from that baseline, normalized by baseline variation.

In [ ]:
baseline_mask = time_s <= 2.0
baseline_mean = rssi_like_db[baseline_mask].mean()
baseline_std = rssi_like_db[baseline_mask].std(ddof=1)

motion_score = np.abs(rssi_like_db - baseline_mean) / max(baseline_std, 1e-6)
motion_threshold = 1.5
motion_detected = motion_score > motion_threshold

baseline_mean, baseline_std, motion_score.max(), motion_detected.mean()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(time_s, reflected_distance_m, color="tab:purple")
axes[0].set_ylabel("path length (m)")
axes[0].set_title("Person-reflected path length")

axes[1].plot(time_s, rssi_like_db, color="tab:blue")
axes[1].axhline(baseline_mean, color="0.3", linestyle="--", label="baseline mean")
axes[1].set_ylabel("RSSI-like power (dB)")
axes[1].legend(loc="upper right")

axes[2].plot(time_s, motion_score, color="tab:red")
axes[2].axhline(motion_threshold, color="0.3", linestyle="--", label="threshold")
axes[2].fill_between(time_s, 0, motion_score, where=motion_detected, color="tab:red", alpha=0.2)
axes[2].set_ylabel("motion score")
axes[2].set_xlabel("time (s)")
axes[2].legend(loc="upper right")

fig.tight_layout()
plt.show()

## What This Teaches

- A moving reflector can perturb received signal power without CSI.
- Path-length changes create phase rotation at the wavelength scale.
- RSSI-like values are useful for intuition but collapse the channel into one number.
- The next notebook should add more reflected paths before generating synthetic CSI vectors.